<a href="https://colab.research.google.com/github/Devil-92/100_Days_of_ML/blob/main/ImplementingCrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Types of Cross Validation Includes : <br>
1 . K Fold <br>
2 . Stratified K Fold  <br>
3 . Hold Out <br>
4 . Leave One Out <br>
5. Group K fold

Depending on which algorithm we choose, training and even validation can be very expensive for a dataset which is of this size. In these cases, we can opt for a hold-out based validation

Instead of splitting The data dynamically every time we run a model, my code pre assigns every row in the dataset to a specific cross-validation fold and saves it permanently

In [1]:
import numpy as np
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings("ignore", category= DeprecationWarning)
warnings.filterwarnings("ignore", category= UserWarning)

In [3]:
def dummy_data(num_samples = 100):
  return pd.DataFrame({
      'feature1' : np.random.randint(1 ,100 , 100) ,
      'feature2' : np.random.uniform(0 , 1 , 100) ,
      'target' : np.random.randint(0 , 1 , 100)
  })

### KFold Cross Validation

In [4]:
from sklearn import model_selection

if __name__ == "__main__" :
  df = dummy_data(num_samples=100)
  df['Kfold'] = -1
  df = df.sample(frac = 1).reset_index(drop = True)

  kf = model_selection.KFold(n_splits = 5)

  for fold , (tra_ , val_) in enumerate(kf.split(X = df)) :
    df.loc[val_ , 'Kfold'] = fold

  df.to_csv('train_folds.csv' , index = False)

### Stratified KFold Cross Validation

In [5]:
if __name__ == "__main__" :
  df = dummy_data(num_samples = 100)
  df['kfold'] = -1
  df = df.sample(frac = 1).reset_index(drop = True)

  y = df.target.values

  kf = model_selection.KFold(n_splits = 5)

  for f , (tra_ , val_) in enumerate(kf.split(X = df , y = y)) :
    df.loc[val_ , 'kfold'] = f

  df.to_csv('train_folds.csv' , index = False)

In [6]:
print('If it’s a standard classification problem, choose stratified k-fold blindly')

If it’s a standard classification problem, choose stratified k-fold blindly


### Holdout Cross Validation

In [7]:
print('Hold-out is also used very frequently with time-series data')

Hold-out is also used very frequently with time-series data


In [8]:
if __name__ == "__main__" :
  df = dummy_data()

  df['kfold'] = -1

  df = df.sample(frac = 1).reset_index(drop = True)

  y = df.target.values

  t_ , v_ = model_selection.train_test_split(
      df.index ,
      test_size = 0.2 ,
      train_size = 0.8 ,
      stratify = y
  )

  df['Holdout'] = -1

  df.loc[t_ , 'holdout'] = 0
  df.loc[v_ , 'holdout'] = 1

  df.to_csv('train_folds.csv' , index = False)

### Leave One Out Cross Validation

In many cases, we have to deal with small datasets and creating big validation sets means losing a lot of data for the model to learn. In those cases, we can opt for a type of k-fold cross-validation where k=N, where N is the number of samples in the dataset. This means that in all folds of training, we will be training on all data samples except 1. The number of folds for this type of cross-validation is the same as the number of samples that we have in the dataset

The good thing about regression problems is that we can use all the cross-validation techniques mentioned above for regression problems except for stratified k-fold.However, if you see that the distribution of targets is not consistent, you can use stratified k-fold. To use stratified k-fold for a regression problem, we have first to divide the target into bins, and then we can use stratified k-fold in the same way as for classification problems

### Stratified KFold for Regression

In [9]:
from sklearn import datasets

In [10]:
print('Sturge’s rule: Number of Bins = 1 + log2(N)')

Sturge’s rule: Number of Bins = 1 + log2(N)


In [11]:
def create_fold(data) :
  data['kfold'] = -1

  data = data.sample(frac = 1).reset_index(drop = True)

  num_bins = int(np.floor(1 + np.log2(len(data))))

  data.loc[: , 'bins'] = pd.cut(
      data['target'] ,
      bins = num_bins ,
      labels = False
  )

  kf = model_selection.StratifiedKFold(n_splits = 5)

  for f , (_t , _v) in enumerate(kf.split(X = data , y = data.bins.values)) :
    data.loc[_v , 'kfold'] = f

  data = data.drop('bins' , axis = 1)
  return data

In [12]:
if __name__ == '__main__' :
  X , y = datasets.make_regression(
      n_targets = 1 ,
      n_features = 100 ,
      n_samples = 15000
  )

  df = pd.DataFrame(
      X , columns = [f"{i}_f" for i in range(X.shape[1])]
  )

  df.loc[: , 'target'] = y

  df = create_fold(df)

### Group K Fold Validation

For example, let’s say we have a problem in which we would like to build a model to detect skin cancer from skin images of patients.<br> Our task is to build a binary classifier which takes an input image and predicts the probability for it being benign or malignant. <br>In these kinds of datasets, you might have multiple images for the same patient in the training dataset. <br> So, to build a good cross-validation system here, you must have stratified k-folds, but you must also make sure that patients in training data do not appear in validation data. Fortunately, scikit-learn offers a type of cross-validation known as GroupKFold. Here the patients can be considered as groups. But unfortunately, there is no way to combine GroupKFold with StratifiedKFold in scikit-learn.